<a href="https://colab.research.google.com/github/Antasey/NCAIR-DSA-Group1/blob/main/NCAIR_DSA_SETUP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NCAIR-DSA — LLM Structuring Pipeline
### ASR Text → Full Clinical Note + Keywords → Database

**Owner:** LLM Structuring Team  
**Goal:** Take a raw ASR transcript (already transcribed from Yoruba/Igbo/Hausa audio), run it through the N-ATLaS LLM to produce a **full structured English clinical note**, extract **supporting keywords**, and save both to the patient database.

**Pipeline covered in this notebook:**
1. Setup (install packages, check GPU)
2. Load N-ATLaS LLM (local, 4-bit quantized — free on Colab T4)
3. Define the structuring prompt
4. Call the LLM + parse its JSON output (full note — the CORE deliverable)
5. Extract keywords (supporting sidebar reference — NOT the main output)
6. Combine into one patient record
7. Save to SQLite database (linked to patient ID for traceable history)
8. End-to-end test with sample ASR transcripts

> **Note:** This notebook assumes ASR transcription has already happened (that's the ASR team's module). We start from raw transcript text.

In [ ]:
# Safer non-interactive login (avoids the token being echoed into cell output/logs,
# which is what happened before — that leaked token must be revoked at
# https://huggingface.co/settings/tokens if you haven't already).
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Enter your Hugging Face access token: ")
login(token=hf_token)
del hf_token

Enter your Hugging Face access token: ··········


In [ ]:
# Install required packages (only needs to run once per Colab session)
!pip install -q transformers accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.1 MB/s eta 0:00:00


In [ ]:
import torch

# Confirm GPU is available — if this prints False, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


In [ ]:
# 1. Clone the repository
!git clone https://github.com/Antasey/NCAIR-DSA-Group1

# 2. Move into the project directory
%cd NCAIR-DSA-Group1

# 3. Install all project dependencies
!pip install -r requirements.txt

Cloning into 'NCAIR-DSA-Group1'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 222 (delta 50), reused 19 (delta 19), pack-reused 126 (from 1)
Receiving objects: 100% (222/222), 159.22 KiB | 1.26 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/NCAIR-DSA-Group1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.7 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "NCAIR1/N-ATLaS"

# 4-bit quantization config — keeps the 8B model small enough for a T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (this can take a few minutes on first run)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config
)

print("Model loaded successfully.")

Loading tokenizer...


config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git pull origin main

From https://github.com/Antasey/NCAIR-DSA-Group1
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
!rm -f data/notes.db
!python app.py

Database ready at: /content/drive/MyDrive/NCAIR-DSA/data/notes.db
/content/NCAIR-DSA-Group1/app.py:215: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="NCAIR-DSA: Patient Intake Assistant", theme=theme) as demo:
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
* Running on local URL:  http://127.0.0.1:7860
INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"
* Running on 